# 06B — Qiskit Hardware-Aware QAOA Transpilation Study

**QuantumOptLab · Milestone 6 · Separate exploratory follow-up · Seed 31**

This experiment uses **only the 20 development-validation graphs** and the Phase 4 QAOA parameters. It **never** loads or uses locked-test labels or the frozen-selector predictions. We measure how hardware connectivity and Qiskit compilation level affect gate counts and circuit depth, without claiming physical-device performance.

**Topologies:** unrestricted connectivity, bidirectional line, bidirectional ring. **Transpiler optimization levels:** 0 and 2. Under line/ring, routing can add two-qubit operations; compiler optimizations may remove some. The gate counts are implementation-dependent, not QPU error measurements.

Do not use these findings to revise Milestone 6A's frozen selector or re-label its test as blind.

### 1. Read the existing Phase 4 validation artifacts

Construct circuits solely from saved validation graph edges and ideal optimized angles. The same Qiskit code generates all topology/optimization combinations, so comparisons differ only in the transpiler settings. Save reports under a new `phase06b_*` prefix, away from the holdout experiment.

In [ ]:
from __future__ import annotations
from pathlib import Path
from hashlib import sha256
import json, platform, time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap
import qiskit

SEED = 31
ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks": ROOT = ROOT.parent
assert (ROOT / "notebooks").is_dir()
EXP = ROOT / "reports" / "experiments"
FIG = ROOT / "reports" / "figures"
EXP.mkdir(parents=True,exist_ok=True)
FIG.mkdir(parents=True,exist_ok=True)
PREFIX=f"phase06b_validation_seed{SEED}"
CATALOG_PATH = EXP / f"phase04_full_seed{SEED}_graph_catalog.json"
IDEAL_PATH = EXP / f"phase04_full_seed{SEED}_qaoa_ideal.csv"
assert CATALOG_PATH.is_file() and IDEAL_PATH.is_file(), "Run Phase 4 full mode first"
CAT_DOC = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))
IDEAL = pd.read_csv(IDEAL_PATH)
assert CAT_DOC["seed"] == SEED
assert IDEAL.config_hash.eq(CAT_DOC["config_hash"]).all()
CAT = pd.DataFrame(CAT_DOC["records"])
TEST_IDS = set(CAT.loc[CAT.split=="test","graph_id"])
VAL_IDS = sorted(CAT.loc[CAT.split=="validation","graph_id"])
assert len(VAL_IDS)==20 and len(TEST_IDS)==20
assert set(IDEAL.graph_id).isdisjoint(TEST_IDS)
VALIDATION = IDEAL.loc[IDEAL.graph_id.isin(VAL_IDS)].copy()
assert len(VALIDATION)==40 and VALIDATION.groupby("graph_id").depth_p.nunique().eq(2).all()
print("Qiskit",qiskit.__version__,"| validation graphs",len(VAL_IDS),"| test graphs UNUSED")

### 2. Reconstruct the exact Phase 4 QAOA circuits

Use Hadamard preparation, weighted `RZZ(-gamma * weight)` cost interactions, and `RX(2 beta)` mixer gates. Save the exact logical circuit resources separately from the routed/optimized resources. Every input graph comes from the original immutable catalog.

In [ ]:
CAT_BY_ID={r["graph_id"]:r for r in CAT_DOC["records"]}
GRAPHS={}
for gid in VAL_IDS:
    record=CAT_BY_ID[gid]
    graph=nx.Graph()
    graph.add_nodes_from(range(int(record["n"])))
    graph.add_weighted_edges_from((int(u),int(v),int(w)) for u,v,w in record["edge_list"])
    assert nx.weisfeiler_lehman_graph_hash(graph,edge_attr="weight")==record["fingerprint"]
    GRAPHS[gid]=graph

def qaoa_circuit(g,params,p):
    pars=np.asarray(params,dtype=float)
    assert pars.shape==(2*p,)
    qc=QuantumCircuit(g.number_of_nodes())
    qc.h(range(g.number_of_nodes()))
    for gamma,beta in zip(pars[:p],pars[p:]):
        for u,v,edge in g.edges(data=True):qc.rzz(-float(gamma)*float(edge["weight"]),u,v)
        for qubit in range(g.number_of_nodes()):qc.rx(2*float(beta),qubit)
    qc.measure_all()
    return qc

CIRCUITS={}
for row in VALIDATION.itertuples():
    CIRCUITS[row.graph_id,int(row.depth_p)]=qaoa_circuit(
        GRAPHS[row.graph_id],json.loads(row.params_json),int(row.depth_p))
assert len(CIRCUITS)==40
print("Constructed",len(CIRCUITS),"measured QAOA circuits; depths 1 and 2")

### 3. Transpile for unrestricted, line and ring connectivity

In the constrained topologies, every CX must connect an allowed pair of **physical** qubits. Qiskit may insert routing SWAPs, which are decomposed to basis CX operations. Optimization level 0 supplies an intentionally unoptimized baseline; level 2 shows the effect of stronger automated compilation. Topologies have identical qubit counts, and all layouts and seeds are fixed.

In [ ]:
def coupling_map(n, topology):
    if topology=="unrestricted":return None
    edges=[]
    for i in range(n-1):edges.extend([(i,i+1),(i+1,i)])
    if topology=="ring" and n>2:edges.extend([(n-1,0),(0,n-1)])
    assert topology in ("line","ring")
    return CouplingMap(couplinglist=edges)

TOPOLOGIES=("unrestricted","line","ring")
OPT_LEVELS=(0,2)
RESULT_PATH=EXP/f"{PREFIX}_compiled_resources.csv"
rows=[]
started=time.perf_counter()
for position,((gid,p),logical) in enumerate(sorted(CIRCUITS.items())):
    n=GRAPHS[gid].number_of_nodes()
    for topology in TOPOLOGIES:
        coupling=coupling_map(n,topology)
        for opt_level in OPT_LEVELS:
            routed=transpile(logical,basis_gates=["h","rx","rz","cx"],
                             coupling_map=coupling, optimization_level=opt_level,
                             seed_transpiler=SEED)
            ops=dict(routed.count_ops())
            assert "rzz" not in ops and "swap" not in ops
            assert int(routed.num_qubits)==n
            if coupling is not None:
                allowed=set(coupling.get_edges())
                for instruction in routed.data:
                    if instruction.operation.name == "cx":
                        endpoints=tuple(routed.find_bit(q).index for q in instruction.qubits)
                        assert endpoints in allowed, f"Illegal CX edge {endpoints} for {topology}"
            cx=int(ops.get("cx",0))
            rows.append({"graph_id":gid,"n":n,"family":CAT_BY_ID[gid]["family"],
                "depth_p":p,"topology":topology,"optimization_level":opt_level,
                "logical_edges":GRAPHS[gid].number_of_edges(),
                "cx_gates":cx,"compiled_depth":int(routed.depth()),
                "operations":int(sum(ops.values())),"measure_count":int(ops.get("measure",0))})
    if (position+1)%10==0:print("Transpiled",position+1,"of",len(CIRCUITS),"logical circuits")
RESULTS=pd.DataFrame(rows)
assert len(RESULTS)==len(CIRCUITS)*len(TOPOLOGIES)*len(OPT_LEVELS)
assert not RESULTS.duplicated(["graph_id","depth_p","topology","optimization_level"]).any()
RESULTS.to_csv(RESULT_PATH,index=False)
print("Saved",len(RESULTS),"Qiskit compiled circuits in",round(time.perf_counter()-started,2),"seconds")
display(RESULTS.groupby(["topology","optimization_level","depth_p"])[["cx_gates","compiled_depth"]].mean().round(2))

### 4. Quantify routing overhead and create compact figures

Use each graph/depth's unrestricted level-0 compilation as the reference. Compare additional CX gates, total CX count and compiled depth across connectivities. This is a resource study on **validation** graphs; changes are not fed back into the frozen M6A selector.

In [ ]:
BASE = RESULTS.loc[(RESULTS.topology=="unrestricted")&(RESULTS.optimization_level==0),
                   ["graph_id","depth_p","cx_gates","compiled_depth"]].rename(columns={
                       "cx_gates":"base_cx","compiled_depth":"base_depth"})
R = RESULTS.merge(BASE,on=["graph_id","depth_p"],validate="many_to_one")
R["cx_overhead"]=R.cx_gates-R.base_cx
R["depth_overhead"]=R.compiled_depth-R.base_depth
R.to_csv(EXP/f"{PREFIX}_resource_overhead.csv",index=False)
agg = R.groupby(["topology","optimization_level","depth_p"],as_index=False).agg(
    mean_cx=("cx_gates","mean"),mean_cx_overhead=("cx_overhead","mean"),
    mean_depth=("compiled_depth","mean"))
agg.to_csv(EXP/f"{PREFIX}_summary.csv",index=False)
display(agg.round(2))
fig,axes=plt.subplots(1,2,figsize=(10.6,3.8))
for level in OPT_LEVELS:
    for p in (1,2):
        seq=agg[(agg.optimization_level==level)&(agg.depth_p==p)].set_index("topology").reindex(TOPOLOGIES)
        axes[0].plot(range(3),seq.mean_cx,marker="o",label=f"p={p}, opt={level}")
        axes[1].plot(range(3),seq.mean_depth,marker="o",label=f"p={p}, opt={level}")
for ax,title,ylabel in [(axes[0],"Compiled CX Count","Mean CX operations"),
                        (axes[1],"Compiled Circuit Depth","Mean circuit depth")]:
    ax.set_xticks(range(3),["All-to-all","Line","Ring"])
    ax.set_title(title);ax.set_ylabel(ylabel);ax.grid(alpha=.2)
axes[0].legend(fontsize=8)
fig.tight_layout()
FIG_PATH=FIG/f"{PREFIX}_topology_resources.png"
fig.savefig(FIG_PATH,dpi=175,bbox_inches="tight")
plt.show()
print("Saved figure:",FIG_PATH)

### 5. Verify compilation and save the hardware-study manifest

All 240 compilations must use only development-validation circuits. We inspect every CX edge against the target device connectivity, check that the output is in the declared basis, and report actual resources without inferring noise fidelity. Successful integrity checks do **not** imply a tested real-processor implementation.

In [ ]:
CHECKS={
    "Only 20 development validation graphs compiled":set(RESULTS.graph_id)==set(VAL_IDS) and len(VAL_IDS)==20,
    "No locked-test graphs compiled":set(RESULTS.graph_id).isdisjoint(TEST_IDS),
    "All 40 logical QAOA circuits reconstructed":len(CIRCUITS)==40,
    "All topology and optimization combinations present":len(RESULTS)==240,
    "No duplicate compilation records":not RESULTS.duplicated(["graph_id","depth_p","topology","optimization_level"]).any(),
    "CX operations and depths nonnegative":RESULTS.cx_gates.ge(0).all() and RESULTS.compiled_depth.gt(0).all(),
    "All measurement registers retained":(RESULTS.measure_count==RESULTS.n).all(),
    "Unrestricted level-0 matches Phase 4 CX formula":all(
        int(r.cx_gates)==2*int(r.logical_edges)*int(r.depth_p)
        for r in RESULTS.loc[(RESULTS.topology=="unrestricted")&(RESULTS.optimization_level==0)].itertuples()),
    "Saved resource tables and figure":RESULT_PATH.exists() and FIG_PATH.exists(),
    "Input source hashes unchanged":CATALOG_PATH.is_file() and IDEAL_PATH.is_file(),
}
CHECKS = {name: bool(value) for name, value in CHECKS.items()}
for name,ok in CHECKS.items():print(("PASS" if ok else "FAIL")+" | "+name)
assert all(CHECKS.values()),"Hardware-study checks failed"
manifest={
    "project":"QuantumOptLab","phase":"6B","scope":"validation-only exploratory",
    "source_catalog_sha256":sha256(CATALOG_PATH.read_bytes()).hexdigest(),
    "source_qaoa_ideal_sha256":sha256(IDEAL_PATH.read_bytes()).hexdigest(),
    "qiskit_version":qiskit.__version__,"python_version":platform.python_version(),
    "topologies":TOPOLOGIES,"optimization_levels":OPT_LEVELS,
    "graphs":len(VAL_IDS),"compiled_circuits":len(RESULTS),"checks":CHECKS,
    "caveats":["No real hardware calibration or job execution",
        "This validation-only follow-up cannot be used to retune the already-frozen M6A policy",
        "Transpiled gate count depends on Qiskit version and heuristic seed"],
}
MANIFEST_PATH=EXP/f"{PREFIX}_manifest.json"
MANIFEST_PATH.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print(f"\nPASS: {sum(CHECKS.values())}/{len(CHECKS)} Phase 6B integrity checks")
print("Research status: 6A = blind locked evaluation; 6B = exploratory validation-only circuit analysis")